In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Consistency Evaluation - Binary Checklist

This notebook evaluates whether the research project at `/net/scratch2/smallyan/leela_eval` meets its stated goals according to the following criteria:

- **CS1**: Conclusion vs Original Results
- **CS2**: Implementation Follows the Plan
- **CS3**: Effect Size
- **CS4**: Justification of Steps and Intermediate Conclusions
- **CS5**: Statistical Significance Reporting

In [2]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")

CUDA available: True
CUDA device: NVIDIA A40
Device count: 1


In [3]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/leela_eval'
for root, dirs, files in os.walk(repo_path):
    # Skip hidden directories
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

leela_eval/
  lc0.onnx
  plan.md
  documentation.pdf
  .gitmodules
  pyproject.toml
  lc0-original.onnx
  768x15x24h-t82-swa-7464000.pb
  .gitignore
  CodeWalkthrough.md
  768x15x24h-t82-swa-7464000.pb.gz
  doc_only_evaluation/
    consistency_evaluation.json
    generalization_eval_summary.json
    code_critic_evaluation.ipynb
    code_critic_summary.json
    self_matching.ipynb
    generalization_eval.ipynb
  iteration_model/
    interesting_puzzles.pkl
    lc0.onnx
    lc0-random.onnx
    LD2.onnx
    unfiltered_puzzles.pkl
    lc0-original.onnx
  lc0_bin/
    lc0.tar.gz
  src/
    leela_logit_lens/
      __init__.py
      tournament/
        logit_lens_engine.py
        constants.py
        __pycache__/
          logit_lens_engine.cpython-312.pyc
          constants.cpython-311.pyc
          logit_lens_engine.cpython-311.pyc
      tools/
        evaluate_puzzles.py
        plotting_helpers.py
        utils.py
        sample_positions.py
        evaluate_concepts.py
        puzzle_h

      __pycache__/
        utils.cpython-311.pyc
        __init__.cpython-312.pyc
        __init__.cpython-311.pyc
        utils.cpython-312.pyc
      engines/
        stockfish_engine.py
        engine.py
        lc0_engine.py
        __pycache__/
          engine.cpython-311.pyc
          lc0_engine.cpython-311.pyc
          stockfish_engine.cpython-311.pyc
          engine.cpython-312.pyc
  Figures/
    Puzzles/
      puzzle_tables_8393.tex
  data/
    eco_openings.pgn
    puzzles.csv
    cclr/
      test/
        123.pgn
        234.pgn
        17.pgn
        5.pgn
        248.pgn
        158.pgn
        2.pgn
        189.pgn
        124.pgn
        233.pgn
        10.pgn
        156.pgn
        187.pgn
        241.pgn
        62.pgn
        19.pgn
        151.pgn
        180.pgn
        246.pgn
        65.pgn
        118.pgn
        164.pgn
        50.pgn
        81.pgn
        163.pgn
        57.pgn
        86.pgn
        208.pgn
        59.pgn
        88.pgn
        206.pgn
    

    replications/
      no_exe_evaluation_replication.md
      self_replication_evaluation.json
  stockfish-8-linux/
  notebooks/
    demo.ipynb
    puzzle_results.ipynb
    figure1.ipynb
    forgotten_puzzle_figure.ipynb
    policy_metrics.ipynb
    tournament_results.ipynb
  results/
    puzzle_accuracy_by_layer.png
    puzzle_results.csv


## Step 1: Read and Understand the Plan

Let's first read the plan file to understand the project goals.

In [4]:
# Read the plan file
plan_path = os.path.join(repo_path, 'plan.md')
with open(plan_path, 'r') as f:
    plan_content = f.read()
print(plan_content)

# Plan
## Objective
Investigate how neural networks progressively build understanding across layers by extending the logit lens technique to analyze the policy network of Leela Chess Zero, examining whether representations are refined through smooth gradual processes or more complex computational mechanisms involving iterative inference with distinct phases.

## Hypothesis
1. Neural networks perform iterative inference with capability progression occurring in distinct computational phases rather than smooth gradual refinement
2. Leela's inference process combines algorithmic computation with learned heuristic priors, where safety-oriented heuristics can override tactical solutions

## Methodology
1. Extend logit lens to Post-LN transformer architectures by applying zero ablation to sublayer outputs beyond layer ℓ while preserving subsequent layer normalizations and ablating layer normalization biases
2. Analyze T82-768x15x24h transformer model with 15 layers and 768-dimensional embeddi

## Step 2: Read the Code Walkthrough and Documentation

Let's read the code walkthrough to understand the implementation.

In [5]:
# Read the Code Walkthrough
walkthrough_path = os.path.join(repo_path, 'CodeWalkthrough.md')
with open(walkthrough_path, 'r') as f:
    walkthrough_content = f.read()
print(walkthrough_content)

# Iterative Inference in a Chess-Playing Neural Network

## Setup

First, install the necessary Python packages:
```bash
pip install -e .
```

Next, download the required data and model files. (If you can't download the following model. You can find model in iteration_model/)

> **📦 All-in-One Download**: For convenience, we've compiled all necessary files into a single Figshare repository: https://figshare.com/s/5342980a9ba8b26985a9. This includes models, datasets, and pre-computed results so you can skip directly to analysis if desired.

### Models

Download the Leela Chess Zero models from the "Evidence of Learned Look-Ahead" paper here: https://figshare.com/s/adc80845c00b67c8fce5 (also available in our all-in-one Figshare above).

Place the model files in your root working directory. For our experiments, we primarily used `lc0-original.onnx`, which is not finetuned and uses position history. The code also works with their finetuned model, `lc0.onnx`, with similar results.

### Data

## Step 3: Read the Implementation Notebooks

Let's examine the implementation notebooks to understand what experiments were actually run.

In [6]:
# List all notebooks
notebooks_path = os.path.join(repo_path, 'notebooks')
notebooks = os.listdir(notebooks_path)
print("Notebooks:", notebooks)

Notebooks: ['demo.ipynb', 'puzzle_results.ipynb', 'figure1.ipynb', 'forgotten_puzzle_figure.ipynb', 'policy_metrics.ipynb', 'tournament_results.ipynb']


In [7]:
# Check the results directory
results_path = os.path.join(repo_path, 'results')
results_files = os.listdir(results_path)
print("Results files:", results_files)

Results files: ['puzzle_accuracy_by_layer.png', 'puzzle_results.csv']


In [8]:
import pandas as pd

# Read puzzle results CSV
puzzle_results = pd.read_csv(os.path.join(results_path, 'puzzle_results.csv'))
print("Puzzle results shape:", puzzle_results.shape)
print("\nColumns:", puzzle_results.columns.tolist())
print("\nFirst few rows:")
puzzle_results.head()

Puzzle results shape: (10, 8)

Columns: ['PuzzleId', 'Rating', 'PGN', 'Solution', 'FEN', 'Moves', 'principal_variation', 'solved_by_layer']

First few rows:


,PuzzleId,Rating,PGN,Solution,FEN,Moves,principal_variation,solved_by_layer
0,00MTG,669,1. e4 e5 2. Nf3 Nc6 3. Bc4 Nf6 4. Nc3 Be7 5. O...,Bf2+ Rxf2 Rxf2 Kxf2,4r1k1/2p1qpp1/3p4/1p1P2PQ/1P5b/3R3P/2PBr3/5RK1...,h4f2 f1f2 e2f2 g1f2,"['f1f2', 'e2f2', 'g1f2']","{0: False, 1: True, 2: True, 3: True, 4: True,..."
1,00Msq,1932,1. e4 e5 2. Nf3 Nc6 3. Bc4 Bc5 4. c3 Bb6 5. O-...,Kf8 Bc4 Qxc4 Nxc4,r5k1/1pp2Bp1/5n1p/1q2N3/3P4/7P/5PP1/4Q1K1 b - ...,g8f8 f7c4 b5c4 e5c4,"['f7c4', 'b5c4', 'e5c4']","{0: False, 1: False, 2: False, 3: False, 4: Fa..."
2,00Pbs,2106,1. d4 Nf6 2. Nf3 d5 3. g3 c5 4. Bg2 e6 5. c3 N...,Qxe7 Bg2+ Ke1 Nhf3#,3r1rk1/Q3qppp/8/1ppb4/2Pn1B1n/2N3P1/PP3P2/R2R1...,a7e7 d5g2 f1e1 h4f3,"['d5g2', 'f1e1', 'h4f3']","{0: False, 1: False, 2: False, 3: False, 4: Fa..."
3,00SIq,1880,1. e4 e6 2. Nf3 d5 3. exd5 exd5 4. Nc3 Nf6 5. ...,Rxf7 Qxf7 Qxf7+ Kxf7,r3r1k1/1Q3ppp/8/pP6/2q5/7P/3R2P1/5R1K w - - 0 30,f1f7 c4f7 b7f7 g8f7,"['c4f7', 'b7f7', 'g8f7']","{0: False, 1: False, 2: True, 3: True, 4: True..."
4,00j6z,2225,1. e4 e5 2. Nf3 Nc6 3. Bc4 Nf6 4. Nc3 Bc5 5. d...,Nxd5 Rxg4+ Qg5+ Rxg5+,r4r2/ppp1qpk1/3p1n2/2bNp3/2B1P1pR/3P2B1/PPPK1P...,f6d5 h4g4 e7g5 g4g5,"['h4g4', 'e7g5', 'g4g5']","{0: False, 1: False, 2: True, 3: True, 4: True..."


In [9]:
# Check the actual number of puzzle results 
print(f"Total puzzles in results: {len(puzzle_results)}")

# The puzzle_results notebook shows 10000 rows - let me check again
# Actually this is a small sample - let me check if there's more data
import ast

# Calculate solve rates per layer
solved_by_layer_dicts = []
for item in puzzle_results['solved_by_layer']:
    if isinstance(item, str):
        solved_by_layer_dicts.append(ast.literal_eval(item))
    else:
        solved_by_layer_dicts.append(item)

max_layer = max(max(d.keys()) for d in solved_by_layer_dicts)
print(f"Max layer: {max_layer}")

layer_solve_rates = {}
for layer in range(max_layer + 1):
    solved = sum(1 for d in solved_by_layer_dicts if d.get(layer, False))
    layer_solve_rates[layer] = solved / len(puzzle_results)
    
print(f"\nSolve rates by layer (sample of {len(puzzle_results)} puzzles):")
for layer, rate in layer_solve_rates.items():
    print(f"Layer {layer}: {rate:.1%}")

Total puzzles in results: 10
Max layer: 15

Solve rates by layer (sample of 10 puzzles):
Layer 0: 0.0%
Layer 1: 10.0%
Layer 2: 30.0%
Layer 3: 40.0%
Layer 4: 40.0%
Layer 5: 40.0%
Layer 6: 40.0%
Layer 7: 30.0%
Layer 8: 40.0%
Layer 9: 40.0%
Layer 10: 50.0%
Layer 11: 50.0%
Layer 12: 40.0%
Layer 13: 60.0%
Layer 14: 80.0%
Layer 15: 90.0%


In [10]:
# The puzzle_results.csv seems to be a small sample. Let me check notebooks for larger dataset references
# The puzzle_results.ipynb notebook shows total_puzzles: 10000 in its output

# Let me check if there are tournament result files
import os

# Check what tournament files exist
for f in os.listdir(repo_path):
    if 'tournament' in f.lower() or f.endswith('.pgn'):
        print(f)
        
# Also check results folder for tournament_games.pgn
results_files = os.listdir(os.path.join(repo_path, 'results'))
print("\nResults directory:", results_files)


Results directory: ['puzzle_accuracy_by_layer.png', 'puzzle_results.csv']


## Step 4: Reading the Forgotten Puzzle Figure Notebook

Let me check the forgotten puzzle notebook which shows the solution discovery and forgetting analysis.

In [11]:
# Read the existing doc_only_evaluation self_matching.ipynb to understand the format
doc_only_eval_path = os.path.join(repo_path, 'doc_only_evaluation')
eval_files = os.listdir(doc_only_eval_path)
print("Doc only evaluation files:", eval_files)

Doc only evaluation files: ['consistency_evaluation.json', 'generalization_eval_summary.json', 'code_critic_evaluation.ipynb', 'code_critic_summary.json', 'self_matching.ipynb', 'generalization_eval.ipynb']


In [12]:
# Check if there's a documentation.pdf in the repo
import subprocess
doc_pdf_path = os.path.join(repo_path, 'documentation.pdf')
print(f"Documentation PDF exists: {os.path.exists(doc_pdf_path)}")

Documentation PDF exists: True
